# 运行基线 SFT

数据准备和训练配置已在 03.02 中详细解析。现在启动 Wordle SFT 训练。

---

## 1. Wordle 1-step smoke

先用正式 `sft_qwen3_1_7b_wordle` recipe 跑 1 个 optimizer step。该步骤会加载 Qwen3-1.7B 权重和 Wordle parquet，并走完两个 rank 的 FSDP、forward、backward 与参数更新；`checkpoint.load_only` 让 smoke 不保存 checkpoint。


In [ ]:
import os
original_dir = original_dir if 'original_dir' in globals() else os.getcwd()
%cd /mnt/workspace/torchtitan-npu
%env BASH_ENV=/home/developer/Ascend/cann/set_env.sh

In [ ]:
%%bash
set -euo pipefail
NGPU=2 \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle \
bash scripts/run_train.sh \
  --dump_folder ./outputs/wordle_smoke \
  --checkpoint.load_only \
  --training.steps 1 \
  --training.global_batch_size 4 \
dataloader:chat_data_loader_config \
  --dataloader.dataset_path "assets/data/wordle" \


成功判据：日志显示两个 rank、`dp_shard=2`、local batch 2、global batch 4、gradient accumulation 1，并完成 step 1；loss 与 grad norm 均为有限值。

---

## 2. 运行 20-step SFT

smoke 通过后，使用 recipe 的正式 global batch 64 和 steps 20 训练，并保存 checkpoint。


In [ ]:
%%bash
set -euo pipefail
# 清理 stale SFT 输出目录，确保 --checkpoint.initial_load_in_hf
# 真正从 assets/hf/Qwen3-1.7B 加载基座权重，而不是从旧 checkpoint 恢复。
# 安全：只删 SFT checkpoint 输出目录，不会动 HF 基座权重。
rm -rf outputs/checkpoint_wordle_sft
echo "已删除 outputs/checkpoint_wordle_sft（如果它存在）。"


In [ ]:
%%bash
set -euo pipefail

CKPT_DIR="outputs/checkpoint_wordle_sft"
if [ -d "$CKPT_DIR" ] && [ -n "$(ls -A "$CKPT_DIR" 2>/dev/null)" ]; then
  echo "错误：$CKPT_DIR 已存在。" >&2
  echo "  --checkpoint.initial_load_in_hf 会被忽略；torchtitan 会尝试" >&2
  echo "  从 $CKPT_DIR/step-* 恢复（若该目录是 HF-only 导出则会崩溃）。" >&2
  echo "  请先运行上方的清理单元，再重新运行本单元。" >&2
  exit 1
fi

ls ./assets/data/wordle
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle \
NGPU=2 \
bash scripts/run_train.sh \
  --hf_assets_path "assets/hf/Qwen3-1.7B" \
  --checkpoint.folder checkpoint_wordle_sft \
  --checkpoint.last_save_in_hf \
  --checkpoint.enable \
  --checkpoint.initial_load_in_hf \
dataloader:chat_data_loader_config \
  --dataloader.dataset_path "assets/data/wordle" \



| 参数 | 含义 |
|------|------|
| `NGPU=2` | 在 A3 环境的两个可见 die 上各启动一个进程，对应两个 rank |
| `MODULE=torchtitan_npu.models.qwen3` | 使用 Qwen3 模型 |
| `CONFIG=sft_qwen3_1_7b_wordle` | 使用 Wordle SFT 配置 |
| `--checkpoint.folder` | 固定本课程 checkpoint 根目录，避免与其他运行混淆 |
| `--checkpoint.last_save_in_hf` | 最后一步额外导出 HuggingFace safetensors 权重布局 |
| `DATASET_PATH` | 使用第 2 章下载到当前仓库的固定数据 revision |

> 关于各配置参数的含义和设计理由（local_batch_size / global_batch_size / gradient accumulation / lr_scheduler / activation_checkpoint / FSDP2），请回顾 **03.02 第 4 节**的详细讲解。

---

## 训练日志解读

训练完成后，先核对初始化日志中的实际配置，再按下面的标准阅读 20 step 日志：



| 指标 | 直接判读标准 |
|------|--------------|
| `loss` | 所有 step 都是有限值，前后窗口的中位数下降或保持稳定，说明这组数据上存在学习信号；短期波动属于正常现象。 |
| `grad_norm` | 裁剪前范数应为有限值，随训练逐步收敛或在稳定区间波动；出现 `NaN`/`Inf`、连续数量级增长，通常对应学习率过大、数据异常或数值溢出。 |
| `tps` / MFU / 显存 | 记录本次命令的吞吐和显存基线；MFU 偏低时同时检查数据等待、通信同步和算子计算，不直接归因于单一瓶颈。 |

> 下面保存的 loss/grad_norm 数字来自历史运行记录；由于当前 recipe、converter 和环境版本可能已变化，它们不是本次代码状态的验收结果。重新运行后，应从同一日志提取 step 1/20 的值，并确认配置摘要与当前 recipe 一致。

---

## 补齐 HF assets

`last_save_in_hf` 导出模型权重布局；推理目录还需要与基座模型匹配的 `config.json` 和 tokenizer 文件。复制这些配套资产，供下一节的推理服务器读取：


In [ ]:
%%bash
set -euo pipefail
src=assets/hf/Qwen3-1.7B
dst=outputs/checkpoint_wordle_sft/step-20
test -d "$dst"
for name in config.json tokenizer.json tokenizer_config.json; do
  test -f "$src/$name"
  cp "$src/$name" "$dst/$name"
done
for name in generation_config.json vocab.json merges.txt; do
  if test -f "$src/$name"; then cp "$src/$name" "$dst/$name"; fi
done
echo "Loadable checkpoint:"
find "$dst" -maxdepth 1 -type f -printf '%f\n' | sort



## 练习

1. （单选题）`--checkpoint.last_save_in_hf` 与配套资产的职责是什么？
    A. 导出 HF 权重布局；config 与 tokenizer 文件由基座资产补齐
    B. 自动导出并验证所有推理文件
    C. 只保存 tokenizer，不保存权重
    D. 把模型转换成 ONNX

2. （单选题）日志中裁剪前 `grad_norm=96.4`、`max_norm=1.0` 且后续 step 为有限值，最合适的解释是什么？
    A. 裁剪前范数较大，训练仍可通过梯度裁剪继续更新
    B. 训练必然已经数值溢出
    C. 说明模型一定学会了输出格式
    D. 说明 optimizer 没有执行

3. （多选题）判断一次 20-step 训练是否产生了稳定学习信号，应检查哪些现象？
    A. loss 和 grad_norm 全部为有限值
    B. loss 在前后窗口的整体趋势下降或进入稳定区间
    C. 是否出现 `NaN`、`Inf` 或连续数量级增长
    D. 只看某一个 step 的 MFU 数字

In [ ]:
%cd $original_dir
!cat ./answer/03.03_answer.txt